In [2]:
import pandas as pd

In [3]:
output_path = "../f0_OSMOSYS_ECU_Output.csv"
#output_path = "../docs/PLANMICCOutput.csv"
current = pd.read_csv(output_path)
current

,Strategy,Future.ID,Fuel,Technology,Emission,Year,Demand,NewCapacity,AccumulatedNewCapacity,TotalCapacityAnnual,...,Capex2023,FixedOpex2023,VarOpex2023,Opex2023,Externalities2023,Capex_GDP,FixedOpex_GDP,VarOpex_GDP,Opex_GDP,Externalities_GDP
0,BAU,0,CEM_PROD,PROD_CEM,NaN,2010,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DDP70,0,CEM_PROD,PROD_CEM,NaN,2010,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,DDP,0,CEM_PROD,PROD_CEM,NaN,2010,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,PA,0,CEM_PROD,PROD_CEM,NaN,2010,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BAU,0,CEM_PROD,PROD_CEM,NaN,2011,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11163,PA,0,NaN,NaN,NaN,2069,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11164,BAU,0,NaN,NaN,NaN,2070,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11165,DDP70,0,NaN,NaN,NaN,2070,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11166,DDP,0,NaN,NaN,NaN,2070,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
calibration_data_path = "calibration_data.csv"
calibration_df = pd.read_csv(calibration_data_path)
calibration_df

,Activity,SubCategory,Year,TotalEmission
0,2A,2A,2010,2174.00
1,2A,2A,2012,2373.40
2,2A,2A,2014,2356.50
3,2A,2A,2016,2242.40
4,2A,2A,2018,2358.30
5,2A,2A,2020,1582.00
6,2A,2A,2021,2239.50
7,2A,2A,2022,2198.40
8,2C,2C,2010,31.04
9,2C,2C,2012,35.47


In [5]:
# Years
calibration_years = list(calibration_df['Year'].unique())

In [14]:
model_filters = (current['Year'].isin(calibration_years))&(current['Strategy'] == 'BAU') & (current['Emission'].isin(['CO2e_OTHER', 'CO2e_HFC', 'CO2e_CEM'])) & (~current['AnnualTechnologyEmission'].isna())
model_df = current.loc[model_filters, ['Year', 'Technology', 'Fuel', 'AnnualTechnologyEmission']]
model_df

,Year,Technology,Fuel,AnnualTechnologyEmission
3416,2010,CERAMICS,NaN,0.3181
3424,2012,CERAMICS,NaN,0.5012
3432,2014,CERAMICS,NaN,0.3789
3440,2016,CERAMICS,NaN,0.3872
3448,2018,CERAMICS,NaN,0.2049
...,...,...,...,...
9069,2020,SODA_ASH,NaN,0.0000
9076,2021,SODA_ASH,NaN,0.0012
9077,2021,SODA_ASH,NaN,0.0000
9084,2022,SODA_ASH,NaN,0.0017


In [16]:
# Calibration 1A1
techs = {
    "2A": ["PROD_CEM", "LIME_PROD", "GLASS_PROD","CERAMICS", "SODA_ASH"],
    "2C": ["IRON_STEEL", "LEAD_PROD"],
    "2D": ["LUBRI","PARAFFIN"],
    "2F": ["HFC_use"],
}

def check_calibration(sub_category:str):
    # Model Results
    model_filters = (model_df['Technology'].isin(techs[sub_category]))
    model_output = model_df.loc[model_filters, ['Year', 'Technology', 'AnnualTechnologyEmission']]
    model = model_output.groupby('Year')[['AnnualTechnologyEmission']].agg('sum').reset_index()
    model['AnnualTechnologyEmission'] = model["AnnualTechnologyEmission"] * 1000 # Mt to Gg
    model = model.rename(columns={"AnnualTechnologyEmission":"Model"})

    # check if all specified techs are found in the model
    model_techs = list(model_output['Technology'].unique())
    if len(model_techs) != len(techs[sub_category]):
        missing_techs = list(set(techs[sub_category]) - set(model_techs))
        print("WARNING: The following techs are missing in the model.")
        print(missing_techs)


    # Calibration Results
    calib = calibration_df.loc[calibration_df['SubCategory']==sub_category, ['Year', 'TotalEmission']]
    calib = calib.rename(columns={"TotalEmission":"INGEI"})

    result = pd.merge(model, calib, on='Year', how='inner')
    result['Error'] = (result['Model'] - result['INGEI'])/ result['INGEI'] * 100
    print(result)

In [17]:
print("*** 2A Industria de los Minerales ***")
check_calibration("2A")

*** 2A Industria de los Minerales ***
['PROD_CEM']
   Year  Model   INGEI      Error
0  2010  392.1  2174.0 -81.964121
1  2012  585.8  2373.4 -75.318109
2  2014  471.6  2356.5 -79.987269
3  2016  491.7  2242.4 -78.072601
4  2018  329.2  2358.3 -86.040792
5  2020  234.3  1582.0 -85.189633
6  2021  274.6  2239.5 -87.738334
7  2022  400.6  2198.4 -81.777656


In [18]:
print("*** 2C Industria de los Metales ***")
check_calibration("2C")

*** 2C Industria de los Metales ***
   Year  Model  INGEI         Error
0  2010   31.1  31.04  1.932990e-01
1  2012   35.5  35.47  8.457852e-02
2  2014   40.5  40.58 -1.971414e-01
3  2016   45.9  45.90  1.548023e-14
4  2018   60.4  60.40  0.000000e+00
5  2020   45.4  45.38  4.407228e-02
6  2021   86.6  86.64 -4.616805e-02
7  2022   86.7  86.67  3.461405e-02


In [19]:
print("*** 2D Uso de Productos no Energéticos de combustibles y de solventes ***")
check_calibration("2D")

*** 2D Uso de Productos no Energéticos de combustibles y de solventes ***
['PARAFFIN']
   Year  Model  INGEI       Error
0  2010    0.0   0.29 -100.000000
1  2012    0.0   0.28 -100.000000
2  2014    5.4   5.91   -8.629442
3  2016    5.7   6.08   -6.250000
4  2018    4.9   5.87  -16.524702
5  2020    3.7   3.93   -5.852417
6  2021    3.6   4.15  -13.253012
7  2022    3.0   3.14   -4.458599


In [20]:
print("*** 2F Uso de productos sustitutos de las SAO ***")
check_calibration("2F")

*** 2F Uso de productos sustitutos de las SAO ***
   Year   Model   INGEI     Error
0  2010   214.9   214.9  0.000000
1  2012   289.1   289.1  0.000000
2  2014   374.5   374.5  0.000000
3  2016   472.9   472.9  0.000000
4  2018   519.1   519.1  0.000000
5  2020  1203.0  1203.0  0.000000
6  2021  1325.1  1325.0  0.007547
7  2022  1957.6  1957.6  0.000000
